In [3]:
import requests
import pandas as pd
import time

def fetch_klines(symbol="ETHUSDT", interval="1m", start_ms=None, end_ms=None):
    """
    Fetch historical klines from Binance (spot).
    Handles pagination automatically.
    """
    url = "https://api.binance.com/api/v3/klines"
    limit = 1000
    all_klines = []

    # If no start time, request earliest available data
    if start_ms is None:
        start_ms = 1502942400000  # Approx earliest Binance spot data (2017)

    while True:
        params = {
            "symbol": symbol,
            "interval": interval,
            "limit": limit,
            "startTime": start_ms
        }

        if end_ms:
            params["endTime"] = end_ms

        r = requests.get(url, params=params)
        data = r.json()

        if not data or len(data) == 0:
            break

        all_klines.extend(data)

        # Move to next batch
        last_open_time = data[-1][0]  # ms timestamp
        start_ms = last_open_time + 1

        # Binance rate limit protection
        time.sleep(0.4)

        # Stop if we reached the end
        if len(data) < limit:
            break

    return all_klines

def klines_to_dataframe(klines):
    columns = [
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_volume", "num_trades",
        "taker_base", "taker_quote", "ignore"
    ]
    df = pd.DataFrame(klines, columns=columns)

    # Convert numeric types
    numeric_cols = ["open", "high", "low", "close", "volume", "quote_volume", "taker_base", "taker_quote"]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)

    # Convert time
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms")

    return df

# ============================
# Download ETHUSDT spot dataset
# ============================

interval = "4h"  # Change to 5m, 15m, 1h, etc.

klines = fetch_klines("ETHUSDT", interval)
df = klines_to_dataframe(klines)

df.to_csv(f"ETHUSDT_{interval}.csv", index=False)
print(f"Saved {len(df)} rows to ETHUSDT_{interval}.csv")


Saved 18134 rows to ETHUSDT_4h.csv
